# N06 — Leakage-Safe Mixed Scikit-learn Pipeline

**Sessions:** 41–44  
**Objective:** Build a mixed-type pipeline, use group-aware validation, and generate a validated submission.

## Task contract
Run top-to-bottom from a fresh kernel. Do not install packages inside the notebook. Record one error or misconception and complete the independent transfer before submission.


In [ ]:
from __future__ import annotations
import hashlib, json, platform, random
from pathlib import Path
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Python:", platform.python_version())
print("Working directory:", Path.cwd())


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

rng=np.random.default_rng(SEED); n=500
df=pd.DataFrame({
 'id':[f'P{i:04d}' for i in range(n)],
 'school':rng.choice([f'S{i:02d}' for i in range(12)],n),
 'device':rng.choice(['desktop','phone','tablet'],n),
 'hours':rng.gamma(2,2,n),
 'late_ratio':rng.beta(1.5,5,n),
})
logit=-.8-.25*df['hours']+2.0*df['late_ratio']+(df['device']=='phone')*.3
df['target']=rng.binomial(1,1/(1+np.exp(-logit)))
df.loc[rng.choice(df.index,20,replace=False),'hours']=np.nan

features=['device','hours','late_ratio']; num=['hours','late_ratio']; cat=['device']
split=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED)
tr,va=next(split.split(df[features],df.target,groups=df.school))
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
pipe=Pipeline([('pre',pre),('model',LogisticRegression(max_iter=1000))])
pipe.fit(df.loc[tr,features],df.loc[tr,'target'])
proba=pipe.predict_proba(df.loc[va,features])[:,1]
print('group overlap:',set(df.loc[tr,'school']) & set(df.loc[va,'school']))
print('F1:',f1_score(df.loc[va,'target'],proba>=.5))
assert not (set(df.loc[tr,'school']) & set(df.loc[va,'school']))


In [ ]:
submission=pd.DataFrame({'id':df.loc[va,'id'].to_numpy(),'prediction':proba})
assert list(submission.columns)==['id','prediction']
assert submission['id'].is_unique
assert submission['prediction'].between(0,1).all()
assert len(submission)==len(va)
print(submission.head())


## Worksheet
1. Where would leakage occur if imputation/scaling were fitted before the split?
2. Why is `handle_unknown='ignore'` necessary?
3. Why should IDs be preserved separately from features?
4. Compare random and group-aware validation.
5. Define the first valid baseline and a stopping rule.


## Independent transfer
Add one justified feature and one alternative model. Keep the split fixed, log runtime and score, validate the output schema, and reject any change that improves only training performance.


## Fresh-run checklist

- [ ] Restart kernel and run all cells in order.
- [ ] Confirm assertions pass.
- [ ] Record package versions and seed.
- [ ] Save the required artifact with a relative path.
- [ ] Add an error-log entry and AI-use note.
- [ ] Explain one teacher-selected cell without reading it.
